# Bike Sharing Demand Prediction with Snowflake ML

**Task:** Regression  
**Dataset:** Kaggle Bike Sharing Demand  
**Models:** Linear Regression, XGBoost  
**Evaluation:** MAE, RMSE, R², RMSLE


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_squared_log_error,
)

import numpy as np
import pandas as pd
import xgboost as xgb


# ============================================================
# CONFIGURATION
# ============================================================

DATABASE = "BIKE_ML_DB"

RAW_SCHEMA = "RAW"
FEATURE_SCHEMA = "FEATURES"
ML_SCHEMA = "ML"

WAREHOUSE = "BIKE_ML_WH"


# ------------------------------------------------------------
# Snowflake tables
# ------------------------------------------------------------

RAW_TABLE = (
    f"{DATABASE}.{RAW_SCHEMA}.BIKE_DEMAND_RAW"
)

FEATURE_TABLE = (
    f"{DATABASE}.{FEATURE_SCHEMA}.BIKE_DEMAND_FEATURES"
)

TRAIN_TABLE = (
    f"{DATABASE}.{ML_SCHEMA}.BIKE_TRAIN"
)

VALIDATION_TABLE = (
    f"{DATABASE}.{ML_SCHEMA}.BIKE_VALIDATION"
)

TEST_TABLE = (
    f"{DATABASE}.{ML_SCHEMA}.BIKE_TEST"
)


# ------------------------------------------------------------
# Feature Store
# ------------------------------------------------------------

FEATURE_STORE_DATABASE = "BIKE_ML_DB"

FEATURE_STORE_NAME = (
    "BIKE_FEATURE_STORE"
)

ENTITY_NAME = "BIKE_HOUR"

FEATURE_VIEW_NAME = (
    "BIKE_DEMAND_FEATURES"
)

FEATURE_VIEW_VERSION = "V1"


# ============================================================
# MODEL FEATURES
# ============================================================
#
# CASUAL and REGISTERED are intentionally excluded.
#
# COUNT = CASUAL + REGISTERED
#
# Therefore using either CASUAL or REGISTERED as a feature
# would leak the target.
#
# DATETIME is also not directly used as a numeric model feature.
# Time-derived features are used instead.
# ============================================================

FEATURE_COLUMNS = [
    "HOUR",
    "DAY",
    "MONTH",
    "YEAR",
    "DAY_OF_WEEK",
    "SEASON",
    "HOLIDAY",
    "WORKINGDAY",
    "WEATHER",
    "TEMP",
    "ATEMP",
    "HUMIDITY",
    "WINDSPEED",
    "IS_WEEKEND",
    "RUSH_HOUR",
]

TARGET_COLUMN = "TARGET_COUNT"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def calculate_metrics(y_true, y_pred):
    """
    Calculate regression metrics.

    RMSLE requires non-negative predictions,
    so predictions are clipped at zero.
    """

    y_pred_safe = np.maximum(
        y_pred,
        0
    )

    mae = mean_absolute_error(
        y_true,
        y_pred_safe
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred_safe
        )
    )

    r2 = r2_score(
        y_true,
        y_pred_safe
    )

    rmsle = np.sqrt(
        mean_squared_log_error(
            y_true,
            y_pred_safe
        )
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "RMSLE": rmsle,
    }


def print_metrics(
    model_name,
    dataset_name,
    metrics,
):
    """
    Print model metrics.
    """

    print(
        f"\n{model_name} - {dataset_name}"
    )

    print(
        "-" * 55
    )

    print(
        f"MAE   : {metrics['MAE']:.4f}"
    )

    print(
        f"RMSE  : {metrics['RMSE']:.4f}"
    )

    print(
        f"R2    : {metrics['R2']:.4f}"
    )

    print(
        f"RMSLE : {metrics['RMSLE']:.4f}"
    )


# ============================================================
# CONNECT TO SNOWFLAKE
# ============================================================

print("\n" + "=" * 70)
print("CONNECTING TO SNOWFLAKE")
print("=" * 70)

session = get_active_session()

session.sql(
    f"USE DATABASE {DATABASE}"
).collect()

session.sql(
    f"USE WAREHOUSE {WAREHOUSE}"
).collect()

print(
    "Connected to Snowflake"
)


# ============================================================
# PROJECT CONTEXT
# ============================================================

print("\nProject context:")

print(
    session.sql(
        """
        SELECT
            CURRENT_ROLE(),
            CURRENT_DATABASE(),
            CURRENT_SCHEMA(),
            CURRENT_WAREHOUSE()
        """
    ).collect()
)


# ============================================================
# RAW DATA
# ============================================================

print("\n" + "=" * 70)
print("RAW DATA")
print("=" * 70)

raw_df = session.table(
    RAW_TABLE
)

raw_count = raw_df.count()

print(
    f"Raw rows: {raw_count}"
)


# ============================================================
# FEATURE TABLE
# ============================================================

print("\n" + "=" * 70)
print("FEATURE TABLE")
print("=" * 70)

feature_df = session.table(
    FEATURE_TABLE
)

feature_count = feature_df.count()

print(
    f"Feature rows: {feature_count}"
)

print(
    "\nFeature table sample:"
)

feature_df.limit(
    5
).show()


# ============================================================
# VERIFY REQUIRED FEATURE COLUMNS
# ============================================================

print(
    "\nChecking required feature columns..."
)

feature_schema_columns = [
    field.name.upper()
    for field in feature_df.schema.fields
]

required_columns = (
    FEATURE_COLUMNS
    + [
        TARGET_COLUMN,
        "DATETIME",
    ]
)

missing_columns = [
    column
    for column in required_columns
    if column not in feature_schema_columns
]

if missing_columns:

    raise ValueError(
        "Missing required columns: "
        f"{missing_columns}"
    )

print(
    "All required feature columns are present."
)


# ============================================================
# SNOWFLAKE FEATURE STORE
# ============================================================

print("\n" + "=" * 70)
print("SNOWFLAKE FEATURE STORE")
print("=" * 70)

feature_store = FeatureStore(
    session=session,
    database=FEATURE_STORE_DATABASE,
    name=FEATURE_STORE_NAME,
    default_warehouse=WAREHOUSE,
)

print(
    "Feature Store connected successfully."
)


# ============================================================
# RETRIEVE FEATURE VIEW
# ============================================================

print(
    "\nRetrieving Feature View..."
)

registered_feature_view = (
    feature_store.get_feature_view(
        FEATURE_VIEW_NAME,
        FEATURE_VIEW_VERSION,
    )
)

print(
    "Feature View:"
)

print(
    f"  Database : {FEATURE_STORE_DATABASE}"
)

print(
    f"  Store    : {FEATURE_STORE_NAME}"
)

print(
    f"  Name     : {FEATURE_VIEW_NAME}"
)

print(
    f"  Version  : {FEATURE_VIEW_VERSION}"
)

print(
    "  Entity / Timestamp : DATETIME"
)


# ============================================================
# CREATE TRAINING DATA SPINE
# ============================================================
#
# We create the spine to demonstrate the Feature Store
# training-data architecture.
#
# We intentionally DO NOT call generate_training_set()
# because the current Feature View definition causes Snowpark
# to expose duplicate DATETIME metadata.
# ============================================================

print("\n" + "=" * 70)
print("CREATING TRAINING DATA SPINE")
print("=" * 70)

spine_df = (
    session
    .table(FEATURE_TABLE)
    .select(
        "DATETIME",
        "TARGET_COUNT",
    )
    .sort(
        "DATETIME"
    )
)

print(
    "Spine sample:"
)

spine_df.limit(
    5
).show()

spine_count = spine_df.count()

print(
    f"Spine rows: {spine_count}"
)


# ============================================================
# FEATURE STORE STATUS
# ============================================================

print("\n" + "=" * 70)
print("FEATURE STORE READY")
print("=" * 70)

print(
    "Feature Store successfully connected."
)

print(
    f"Entity : {ENTITY_NAME}"
)

print(
    f"Feature View : "
    f"{FEATURE_VIEW_NAME}${FEATURE_VIEW_VERSION}"
)

print(
    "Training-set generation skipped because the current "
    "Feature View exposes duplicate DATETIME metadata."
)

print(
    "Model training will use the existing chronological "
    "train / validation / test tables."
)


# ============================================================
# LOAD CHRONOLOGICAL TRAINING DATA
# ============================================================

print("\n" + "=" * 70)
print(
    "LOADING CHRONOLOGICAL "
    "TRAIN / VALIDATION / TEST DATA"
)
print("=" * 70)


train_df = session.table(
    TRAIN_TABLE
)

validation_df = session.table(
    VALIDATION_TABLE
)

test_df = session.table(
    TEST_TABLE
)


# ============================================================
# DATASET COUNTS
# ============================================================

train_count = (
    train_df.count()
)

validation_count = (
    validation_df.count()
)

test_count = (
    test_df.count()
)

print(
    f"Train rows      : {train_count}"
)

print(
    f"Validation rows : {validation_count}"
)

print(
    f"Test rows       : {test_count}"
)

print(
    f"Total rows      : "
    f"{train_count + validation_count + test_count}"
)


# ============================================================
# CHECK CHRONOLOGICAL SPLIT
# ============================================================

print(
    "\nChecking chronological split..."
)


train_min = (
    train_df
    .select("DATETIME")
    .sort("DATETIME")
    .limit(1)
    .collect()[0][0]
)

train_max = (
    train_df
    .select("DATETIME")
    .sort("DATETIME", ascending=False)
    .limit(1)
    .collect()[0][0]
)

validation_min = (
    validation_df
    .select("DATETIME")
    .sort("DATETIME")
    .limit(1)
    .collect()[0][0]
)

validation_max = (
    validation_df
    .select("DATETIME")
    .sort("DATETIME", ascending=False)
    .limit(1)
    .collect()[0][0]
)

test_min = (
    test_df
    .select("DATETIME")
    .sort("DATETIME")
    .limit(1)
    .collect()[0][0]
)

test_max = (
    test_df
    .select("DATETIME")
    .sort("DATETIME", ascending=False)
    .limit(1)
    .collect()[0][0]
)


print(
    f"Train      : {train_min} -> {train_max}"
)

print(
    f"Validation : {validation_min} -> {validation_max}"
)

print(
    f"Test       : {test_min} -> {test_max}"
)


# ============================================================
# CONVERT TO PANDAS
# ============================================================

print(
    "\nConverting Snowflake data to Pandas..."
)


train_pd = (
    train_df
    .select(
        *FEATURE_COLUMNS,
        TARGET_COLUMN,
    )
    .to_pandas()
)

validation_pd = (
    validation_df
    .select(
        *FEATURE_COLUMNS,
        TARGET_COLUMN,
    )
    .to_pandas()
)

test_pd = (
    test_df
    .select(
        *FEATURE_COLUMNS,
        TARGET_COLUMN,
    )
    .to_pandas()
)

print(
    "Pandas conversion complete."
)


# ============================================================
# PREPARE X / Y
# ============================================================

X_train = train_pd[
    FEATURE_COLUMNS
]

y_train = train_pd[
    TARGET_COLUMN
]

X_validation = validation_pd[
    FEATURE_COLUMNS
]

y_validation = validation_pd[
    TARGET_COLUMN
]

X_test = test_pd[
    FEATURE_COLUMNS
]

y_test = test_pd[
    TARGET_COLUMN
]


# ============================================================
# DATASET SHAPES
# ============================================================

print(
    "\nDataset shapes:"
)

print(
    f"X_train       : {X_train.shape}"
)

print(
    f"X_validation  : {X_validation.shape}"
)

print(
    f"X_test        : {X_test.shape}"
)


# ============================================================
# MODEL 1
# LINEAR REGRESSION
# ============================================================

print("\n" + "=" * 70)
print("MODEL 1 - LINEAR REGRESSION")
print("=" * 70)

linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train,
)

print(
    "Linear Regression training complete."
)


# ============================================================
# LINEAR REGRESSION PREDICTIONS
# ============================================================

linear_validation_predictions = (
    linear_model.predict(
        X_validation
    )
)

linear_test_predictions = (
    linear_model.predict(
        X_test
    )
)


# ============================================================
# LINEAR REGRESSION EVALUATION
# ============================================================

linear_validation_metrics = (
    calculate_metrics(
        y_validation,
        linear_validation_predictions,
    )
)

linear_test_metrics = (
    calculate_metrics(
        y_test,
        linear_test_predictions,
    )
)


print_metrics(
    "Linear Regression",
    "Validation",
    linear_validation_metrics,
)

print_metrics(
    "Linear Regression",
    "Test",
    linear_test_metrics,
)


# ============================================================
# MODEL 2
# XGBOOST
# ============================================================

print("\n" + "=" * 70)
print("MODEL 2 - XGBOOST")
print("=" * 70)

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

print(
    "Training XGBoost..."
)

xgb_model.fit(
    X_train,
    y_train,
)

print(
    "XGBoost training complete."
)


# ============================================================
# XGBOOST PREDICTIONS
# ============================================================

xgb_validation_predictions = (
    xgb_model.predict(
        X_validation
    )
)

xgb_test_predictions = (
    xgb_model.predict(
        X_test
    )
)


# ============================================================
# XGBOOST EVALUATION
# ============================================================

xgb_validation_metrics = (
    calculate_metrics(
        y_validation,
        xgb_validation_predictions,
    )
)

xgb_test_metrics = (
    calculate_metrics(
        y_test,
        xgb_test_predictions,
    )
)


print_metrics(
    "XGBoost",
    "Validation",
    xgb_validation_metrics,
)

print_metrics(
    "XGBoost",
    "Test",
    xgb_test_metrics,
)


# ============================================================
# MODEL COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)


comparison = pd.DataFrame(
    [
        {
            "MODEL": "Linear Regression",

            "VALIDATION_MAE":
                linear_validation_metrics[
                    "MAE"
                ],

            "VALIDATION_RMSE":
                linear_validation_metrics[
                    "RMSE"
                ],

            "VALIDATION_R2":
                linear_validation_metrics[
                    "R2"
                ],

            "VALIDATION_RMSLE":
                linear_validation_metrics[
                    "RMSLE"
                ],

            "TEST_MAE":
                linear_test_metrics[
                    "MAE"
                ],

            "TEST_RMSE":
                linear_test_metrics[
                    "RMSE"
                ],

            "TEST_R2":
                linear_test_metrics[
                    "R2"
                ],

            "TEST_RMSLE":
                linear_test_metrics[
                    "RMSLE"
                ],
        },

        {
            "MODEL": "XGBoost",

            "VALIDATION_MAE":
                xgb_validation_metrics[
                    "MAE"
                ],

            "VALIDATION_RMSE":
                xgb_validation_metrics[
                    "RMSE"
                ],

            "VALIDATION_R2":
                xgb_validation_metrics[
                    "R2"
                ],

            "VALIDATION_RMSLE":
                xgb_validation_metrics[
                    "RMSLE"
                ],

            "TEST_MAE":
                xgb_test_metrics[
                    "MAE"
                ],

            "TEST_RMSE":
                xgb_test_metrics[
                    "RMSE"
                ],

            "TEST_R2":
                xgb_test_metrics[
                    "R2"
                ],

            "TEST_RMSLE":
                xgb_test_metrics[
                    "RMSLE"
                ],
        },
    ]
)


print(
    comparison.to_string(
        index=False
    )
)


# ============================================================
# XGBOOST FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("XGBOOST FEATURE IMPORTANCE")
print("=" * 70)


feature_importance = pd.DataFrame(
    {
        "FEATURE":
            FEATURE_COLUMNS,

        "IMPORTANCE":
            xgb_model.feature_importances_,
    }
)


feature_importance = (
    feature_importance
    .sort_values(
        by="IMPORTANCE",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print(
    feature_importance.to_string(
        index=False
    )
)


# ============================================================
# SAMPLE PREDICTIONS
# ============================================================

print("\n" + "=" * 70)
print("SAMPLE XGBOOST PREDICTIONS")
print("=" * 70)


sample_predictions = (
    test_df
    .select(
        "DATETIME",
        TARGET_COLUMN,
    )
    .to_pandas()
)


sample_predictions[
    "PREDICTED_COUNT"
] = xgb_test_predictions


sample_predictions[
    "ABSOLUTE_ERROR"
] = np.abs(
    sample_predictions[
        TARGET_COLUMN
    ]
    -
    sample_predictions[
        "PREDICTED_COUNT"
    ]
)


print(
    sample_predictions
    .head(10)
    .to_string(
        index=False
    )
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)


print("\nDATA")

print(
    f"  Raw rows             : "
    f"{raw_count}"
)

print(
    f"  Feature rows         : "
    f"{feature_count}"
)

print(
    f"  Spine rows           : "
    f"{spine_count}"
)

print(
    f"  Train rows           : "
    f"{train_count}"
)

print(
    f"  Validation rows      : "
    f"{validation_count}"
)

print(
    f"  Test rows            : "
    f"{test_count}"
)


print("\nFEATURE STORE")

print(
    f"  Feature Store        : "
    f"{FEATURE_STORE_NAME}"
)

print(
    f"  Entity               : "
    f"{ENTITY_NAME}"
)

print(
    f"  Feature View         : "
    f"{FEATURE_VIEW_NAME}"
    f"${FEATURE_VIEW_VERSION}"
)

print(
    "  Status               : READY"
)


print("\nMODELS")

print(
    "  Linear Regression    : trained"
)

print(
    "  XGBoost              : trained"
)


print("\nXGBOOST TEST METRICS")

print(
    f"  MAE                  : "
    f"{xgb_test_metrics['MAE']:.4f}"
)

print(
    f"  RMSE                 : "
    f"{xgb_test_metrics['RMSE']:.4f}"
)

print(
    f"  R2                   : "
    f"{xgb_test_metrics['R2']:.4f}"
)

print(
    f"  RMSLE                : "
    f"{xgb_test_metrics['RMSLE']:.4f}"
)


print("\n" + "=" * 70)
print("END")
print("=" * 70)
